[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap09_juros_compostos.ipynb)

# Capítulo 9: Princípios: juros compostos, risco e diversificação

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Juros compostos, a matemática da perda e o peso dos custos, com contas que você pode refazer com os seus números.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Aporte mensal e juros compostos
Todos os valores estão em reais de hoje: a taxa usada é o rendimento **acima da inflação**.

In [ ]:
APORTE = 500            # MUDE AQUI: aporte mensal em R$
TAXA_REAL_ANO = 0.04    # MUDE AQUI: rendimento acima da inflação ao ano
ANOS = 30               # MUDE AQUI

i = (1 + TAXA_REAL_ANO) ** (1/12) - 1
meses = np.arange(0, ANOS*12 + 1)
saldo = APORTE * (((1 + i) ** meses - 1) / i)
aportado = APORTE * meses
fig, ax = plt.subplots()
ax.fill_between(meses/12, aportado, color=CORES['azul_claro'], alpha=0.5, label='O que você colocou')
ax.fill_between(meses/12, aportado, saldo, color=CORES['verde'], alpha=0.5, label='O que os juros fizeram')
ax.set_title(f'{brl(APORTE,0)} por mês a {pct(TAXA_REAL_ANO,1)} ao ano acima da inflação'); ax.set_xlabel('Anos'); eixo_reais(ax); ax.legend(); plt.show()
for a in (10, 20, 30):
    if a <= ANOS:
        print(f'{a} anos: saldo {brl(saldo[a*12])}, sendo {brl(aportado[a*12])} de aportes')

## 2. Começar cedo: Ana e Bruno (Figura 9.1 do livro)
Ana aplica R\$ 300 por mês dos 25 aos 35 anos e para. Bruno aplica R\$ 300 por mês dos 35 aos 65. Os dois ganham 6% ao ano acima da inflação.

In [ ]:
MENSAL = 300        # MUDE AQUI
TAXA = 0.06         # MUDE AQUI
i = (1 + TAXA) ** (1/12) - 1
def evolucao(inicio_aporte, fim_aporte, idade0=25, idade_fim=65):
    s = 0; hist = {}
    for m in range((idade_fim - idade0) * 12):
        idade = idade0 + m / 12
        s *= (1 + i)
        if inicio_aporte <= idade < fim_aporte:
            s += MENSAL
        if (m + 1) % 60 == 0:
            hist[int(idade0 + (m + 1) / 12)] = s
    return pd.Series(hist)
ana = evolucao(25, 35); bruno = evolucao(35, 65)
print(f'Ana aplicou {brl(MENSAL*120)} e chega aos 65 com {brl(ana.iloc[-1])}')
print(f'Bruno aplicou {brl(MENSAL*360)} e chega aos 65 com {brl(bruno.iloc[-1])}')
ax = pd.DataFrame({'Ana (dos 25 aos 35)': ana, 'Bruno (dos 35 aos 65)': bruno}).plot.bar(color=[CORES['verde'], CORES['azul']], rot=0, title='Saldo a cada 5 anos, em reais de hoje')
ax.set_xlabel('Idade'); eixo_reais(ax); plt.show()

## 3. Pequenas diferenças, grandes resultados
R$ 50 mil aplicados por 20 anos, com rendimentos reais diferentes.

In [ ]:
INICIAL = 50_000   # MUDE AQUI
for t in (0.02, 0.04, 0.06, 0.08):
    print(f'{pct(t,0)} ao ano acima da inflação: {brl(INICIAL * (1+t)**20)} depois de 20 anos')

## 4. A matemática da perda
Quem perde 50% precisa ganhar 100% para voltar ao ponto de partida.

In [ ]:
for queda in (0.10, 0.20, 0.30, 0.50, 0.80):
    print(f'Se cair {pct(queda,0)}, precisa subir {pct(1/(1-queda) - 1, 0)} para empatar')

## 5. O inimigo silencioso: os custos
R$ 100 mil por 30 anos, com rendimento bruto de 6% ao ano acima da inflação, e custos anuais diferentes.

In [ ]:
BRUTO = 0.06   # MUDE AQUI
INICIAL = 100_000
for custo in (0.0, 0.005, 0.01, 0.02):
    print(f'Custo de {pct(custo,1)} ao ano: {brl(INICIAL * (1 + BRUTO - custo) ** 30)} depois de 30 anos')

Faça a sua conta na [calculadora de juros compostos](https://www.traderbrasil.com/blog/juros-compostos-o-que-e-como-funciona.php) ou, ao contrário, partindo do objetivo, na [calculadora de meta financeira](https://www.traderbrasil.com/blog/calculadora-meta-financeira-quanto-guardar-por-mes.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 9 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap09/

Achou um erro? Escreva para contato@traderbrasil.com.